In [43]:
import dlt
import requests
import time

import pandas as pd

from datetime import datetime, timezone
from uuid import uuid4

from dlt.destinations import filesystem
from dlt.destinations.adapters import iceberg_adapter, iceberg_partition

In [33]:
SOURCE_SYSTEM = "openlibrary"
INGESTION_TYPE = "full"

INGESTED_AT = datetime.now(timezone.utc)
RUN_ID = str(uuid4())

BRONZE_BUCKET_URL = "s3://book-library-lakehouse/bronze"

destination = filesystem(
    bucket_url=BRONZE_BUCKET_URL
)

iceberg_partition.day("_ingested_at")

PartitionSpec(source_column='_ingested_at', transform='day', param_value=None, partition_field=None)

In [34]:
QUERY = "title:python"
API_URL = "https://openlibrary.org/search.json"
PAGE_SIZE = 100
HEADERS = {
    "User-Agent": "BookLibraryDataPipeline/1.0"
}

@dlt.resource(
    name="books",
    write_disposition="replace"
)

def openlibrary_books():

    offset = 0
    total_found = None

    with requests.Session() as session:

        while total_found is None or offset < total_found:

            params = {
                "q": QUERY,
                "fields": (
                    "key,title,author_name,author_key,"
                    "first_publish_year,edition_count,isbn,language"
                ),
                "limit": PAGE_SIZE,
                "offset": offset
            }

            response = session.get(
                API_URL,
                params=params,
                headers=HEADERS,
                timeout=60
            )

            response.raise_for_status()

            data = response.json()

            if total_found is None:
                total_found = int(data.get("numFound", 0))

                print("Najdenih knjig:", total_found)

                if total_found == 0:
                    raise RuntimeError(
                        "API ni vrnil nobene knjige. "
                        "Preveri query. Replace je bil prekinjen."
                    )

            docs = data.get("docs", [])

            if not docs:
                break

            for book in docs:

                item = dict(book)
                item["api_query"] = QUERY #novi stolpec

                author_names = item.pop("author_name", []) or []
                author_keys = item.pop("author_key", []) or []

                item["authors"] = "; ".join(author_names)
                item["author_ids"] = "; ".join(author_keys)

                item["openlibrary_id"] = item.get("key")

                # metadata
                item["_source_system"] = SOURCE_SYSTEM
                item["_source_resource"] = "books"
                item["_ingestion_type"] = INGESTION_TYPE
                item["_ingested_at"] = INGESTED_AT
                item["_load_id"] = RUN_ID

                yield item

            offset += len(docs)

            print(
                f"Prenesenih {offset} od {total_found} knjig."
            )

            if offset < total_found:
                time.sleep(1.1)

    if offset < total_found:
        raise RuntimeError(
            f"Nalaganje ni bilo popolno: prenesenih {offset} "
            f"od {total_found} rezultatov."
        )

In [35]:
books_resource = openlibrary_books()

iceberg_adapter(
    books_resource,
    partition=[
        iceberg_partition.day("_ingested_at")
    ]
)

<@dlt.resource(name='books', section='__main__', write_disposition='replace')>

In [36]:
pipeline = dlt.pipeline(
    pipeline_name="openlibrary_to_lakehouse",
    destination=destination,
    dataset_name="openlibrary"
)

In [37]:
load_info = pipeline.run(
    books_resource,
    write_disposition="replace",
    table_format="iceberg"
)

print("Open Library - Iceberg BRONZE ... DONE")
print(load_info)
print("RUN ID:", RUN_ID)
print("INGESTED AT:", INGESTED_AT)

Najdenih knjig: 5856
Prenesenih 100 od 5856 knjig.
Prenesenih 200 od 5856 knjig.
Prenesenih 300 od 5856 knjig.
Prenesenih 400 od 5856 knjig.
Prenesenih 500 od 5856 knjig.
Prenesenih 600 od 5856 knjig.
Prenesenih 700 od 5856 knjig.
Prenesenih 800 od 5856 knjig.
Prenesenih 900 od 5856 knjig.
Prenesenih 1000 od 5856 knjig.
Prenesenih 1100 od 5856 knjig.
Prenesenih 1200 od 5856 knjig.
Prenesenih 1300 od 5856 knjig.
Prenesenih 1400 od 5856 knjig.
Prenesenih 1500 od 5856 knjig.
Prenesenih 1600 od 5856 knjig.
Prenesenih 1700 od 5856 knjig.
Prenesenih 1800 od 5856 knjig.
Prenesenih 1900 od 5856 knjig.
Prenesenih 2000 od 5856 knjig.
Prenesenih 2100 od 5856 knjig.
Prenesenih 2200 od 5856 knjig.
Prenesenih 2300 od 5856 knjig.
Prenesenih 2400 od 5856 knjig.
Prenesenih 2500 od 5856 knjig.
Prenesenih 2600 od 5856 knjig.
Prenesenih 2700 od 5856 knjig.
Prenesenih 2800 od 5856 knjig.
Prenesenih 2900 od 5856 knjig.
Prenesenih 3000 od 5856 knjig.
Prenesenih 3100 od 5856 knjig.
Prenesenih 3200 od 5856 knj

In [31]:
from pyiceberg.catalog import load_catalog

catalog = load_catalog(    "book_library",
    **{
        "type": "sql",
        "uri": "sqlite:///./catalog/iceberg_catalog.db",
        "warehouse": "s3://book-library-lakehouse/warehouse",

        "s3.endpoint": "http://127.0.0.1:9000",
        "s3.access-key-id": "nelacopot",
        "s3.secret-access-key": "nelacopot",
        "s3.region": "us-east-1",
        "s3.force-virtual-addressing": "false",

        "py-io-impl": "pyiceberg.io.pyarrow.PyArrowFileIO"
    })

api_table_before = catalog.load_table(
    ("openlibrary", "books")
)

api_snapshot_before = (
    api_table_before.current_snapshot().snapshot_id
)

api_schema_before = api_table_before.schema()

api_files_before = (
    api_table_before.inspect
    .data_files(snapshot_id=api_snapshot_before)
    .to_pylist()
)

print("Začetni snapshot:", api_snapshot_before)
print("Začetni schema ID:", api_schema_before.schema_id)

print("\nZačetna shema:")

for field in api_schema_before.fields:
    print(
        field.field_id,
        field.name,
        field.field_type
    )

print("\nŠtevilo začetnih Parquet datotek:", len(api_files_before))

Začetni snapshot: 7070250783243044529
Začetni schema ID: 0

Začetna shema:
1 edition_count long
2 first_publish_year long
3 key string
4 title string
5 authors string
6 author_ids string
7 openlibrary_id string
8 _source_system string
9 _source_resource string
10 _ingestion_type string
11 _ingested_at timestamptz
12 _load_id string
13 _dlt_load_id string
14 _dlt_id string

Število začetnih Parquet datotek: 1


In [38]:
api_table_after = catalog.load_table(
    ("openlibrary", "books")
)

api_schema_after = api_table_after.schema()

print("Stari schema ID:", api_schema_before.schema_id)
print("Novi schema ID:", api_schema_after.schema_id)

print("\nNova shema:")

for field in api_schema_after.fields:
    print(
        field.field_id,
        field.name,
        field.field_type
    )

api_data_after = api_table_after.scan().to_arrow()

print("Število zapisov:", api_data_after.num_rows)
print("Stolpci:", api_data_after.column_names)

print(
    "Vrednosti api_query:",
    set(api_data_after["api_query"].to_pylist())
)



Stari schema ID: 0
Novi schema ID: 1

Nova shema:
1 edition_count long
2 first_publish_year long
3 key string
4 title string
5 authors string
6 author_ids string
7 openlibrary_id string
8 _source_system string
9 _source_resource string
10 _ingestion_type string
11 _ingested_at timestamptz
12 _load_id string
13 _dlt_load_id string
14 _dlt_id string
15 api_query string
Število zapisov: 5856
Stolpci: ['edition_count', 'first_publish_year', 'key', 'title', 'authors', 'author_ids', 'openlibrary_id', '_source_system', '_source_resource', '_ingestion_type', '_ingested_at', '_load_id', '_dlt_load_id', '_dlt_id', 'api_query']
Vrednosti api_query: {'title:python'}


Iceberg je ohranil tudi prejšnjo različico sheme, samo dodana je nova
Stara Parquet datoteka se je ohranila, dodane so nove

ker pri Open Library uporabljamo polni replace, bo novi trenutni snapshot vseboval datoteke novega polnega vnosa...
- stare datoteke niso bile prepisane in ostajajo povezane s starim snapshotom
- ne pomeni pa, da jih novi trenutni snapshot še uporablja

In [41]:
print("Zgodovina snapshotov Open Library:")

snapshots = (
    api_table_after.inspect.snapshots().to_pylist()
)

for snapshot in snapshots:
    print(
        "\nČas:", snapshot["committed_at"],
        "\nSnapshot ID:", snapshot["snapshot_id"],
        "\nOperacija:", snapshot["operation"],
        "\nNadrejeni snapshot:", snapshot["parent_id"]
    )

Zgodovina snapshotov Open Library:

Čas: 2026-11-03 10:12:50.739000 
Snapshot ID: 7616878257586896298 
Operacija: append 
Nadrejeni snapshot: None

Čas: 2026-11-03 10:42:23.549000 
Snapshot ID: 5847528146634748977 
Operacija: delete 
Nadrejeni snapshot: 7616878257586896298

Čas: 2026-11-03 10:42:23.936000 
Snapshot ID: 7070250783243044529 
Operacija: append 
Nadrejeni snapshot: 5847528146634748977

Čas: 2026-11-03 11:06:44.166000 
Snapshot ID: 8004944244606236377 
Operacija: delete 
Nadrejeni snapshot: 7070250783243044529

Čas: 2026-11-03 11:06:44.561000 
Snapshot ID: 6753391268861399453 
Operacija: append 
Nadrejeni snapshot: 8004944244606236377


In [8]:
from pyiceberg.catalog import load_catalog

catalog = load_catalog(    "book_library",
    **{
        "type": "sql",
        "uri": "sqlite:///./catalog/iceberg_catalog.db",
        "warehouse": "s3://book-library-lakehouse/warehouse",

        "s3.endpoint": "http://127.0.0.1:9000",
        "s3.access-key-id": "nelacopot",
        "s3.secret-access-key": "nelacopot",
        "s3.region": "us-east-1",
        "s3.force-virtual-addressing": "false",

        "py-io-impl": "pyiceberg.io.pyarrow.PyArrowFileIO"
    })

api_table = catalog.load_table(
    ("openlibrary", "books")
)

old_snapshot_id = 7616878257586896298

old_files = (
    api_table.inspect
    .data_files(snapshot_id=old_snapshot_id)
    .to_pylist()
)

current_files = (
    api_table.inspect
    .data_files()
    .to_pylist()
)

def file_signature(files):
    return sorted([
        (
            f["file_path"],
            f["file_size_in_bytes"],
            f["record_count"]
        )
        for f in files
    ])

print("Stare Parquet datoteke:", len(old_files))
print("Trenutne Parquet datoteke:", len(current_files))

print(
    "Stare datoteke so ohranjene v prvotnem snapshotu:",
    bool(old_files)
)

old_paths = {f["file_path"] for f in old_files}
current_paths = {f["file_path"] for f in current_files}

print(
    "Skupno število starih datotek, ki jih uporablja trenutni snapshot:",
    len(old_paths.intersection(current_paths))
)

print(
    "Ali sta nabora datotek enaka:",
    file_signature(old_files) == file_signature(current_files)
)

Stare Parquet datoteke: 1
Trenutne Parquet datoteke: 1
Stare datoteke so ohranjene v prvotnem snapshotu: True
Skupno število starih datotek, ki jih uporablja trenutni snapshot: 0
Ali sta nabora datotek enaka: False
